In [68]:
import numpy as np, pandas as pd, optuna
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import (classification_report, roc_auc_score,
                             average_precision_score, precision_recall_curve)
from lightgbm import LGBMClassifier


optuna.logging.set_verbosity(optuna.logging.WARNING)


In [69]:


def build_pipeline(num_cols, cat_cols, **params):
    prep = ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                          ("sc", StandardScaler())]), num_cols),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                          ("oh", OneHotEncoder(handle_unknown="ignore",
                                               sparse_output=False))]), cat_cols),
    ])
    return Pipeline([("prep", prep),
                     ("clf", LGBMClassifier(verbose=-1, **params))])


def tune(X, y, num_cols, cat_cols, n_trials=40):
    cv = StratifiedKFold(5, shuffle=True, random_state=42)

    def objective(trial):
        p = {
            "n_estimators":  trial.suggest_int("n_estimators", 200, 800, step=100),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            "num_leaves":    trial.suggest_int("num_leaves", 8, 64, log=True),
            "subsample":     trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
            "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1.0, 5.0),
        }
        return cross_val_score(build_pipeline(num_cols, cat_cols, **p),
                               X, y, cv=cv, scoring="average_precision").mean()

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials)
    return study.best_params 


def pick_threshold(y_true, proba, min_recall=0.75):
    """Chọn ngưỡng đảm bảo bắt được min_recall khách rời bỏ."""
    prec, rec, thr = precision_recall_curve(y_true, proba)
    ok = np.where(rec[:-1] >= min_recall)[0]
    i = ok[-1]
    return float(thr[i]), float(prec[i]), float(rec[i])


## Processing

In [70]:
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")
y = (df.pop("Churn") == "Yes").astype(int)

In [71]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65


In [72]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [73]:
blank_total_charges = (df["TotalCharges"].str.strip() == "").sum()

print("Blank TotalCharges:", blank_total_charges)

Blank TotalCharges: 11


In [74]:
df.isnull().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
dtype: int64

In [75]:
# Cek duplicate rows
duplicate_rows = df.duplicated().sum()

print("Duplicate rows:", duplicate_rows)

Duplicate rows: 0


In [76]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

In [77]:
df["TotalCharges"].dtype

dtype('float64')

In [78]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65


In [79]:
df.shape

(7043, 20)

In [80]:
# Check missing values after conversion
print("Missing TotalCharges:", df["TotalCharges"].isnull().sum())

Missing TotalCharges: 11


In [81]:
df["TotalCharges"].isnull()

0       False
1       False
2       False
3       False
4       False
        ...  
7038    False
7039    False
7040    False
7041    False
7042    False
Name: TotalCharges, Length: 7043, dtype: bool

In [82]:
df[df["TotalCharges"].isnull()][
    ["customerID", "tenure", "Contract", "MonthlyCharges", "TotalCharges"]
]

,customerID,tenure,Contract,MonthlyCharges,TotalCharges
488,4472-LVYGI,0,Two year,52.55,NaN
753,3115-CZMZD,0,Two year,20.25,NaN
936,5709-LVOEQ,0,Two year,80.85,NaN
1082,4367-NUYAO,0,Two year,25.75,NaN
1340,1371-DWPAZ,0,Two year,56.05,NaN
3331,7644-OMVMY,0,Two year,19.85,NaN
3826,3213-VVOLG,0,Two year,25.35,NaN
4380,2520-SGTTA,0,Two year,20.00,NaN
5218,2923-ARZLG,0,One year,19.70,NaN
6670,4075-WKNIU,0,Two year,73.35,NaN


In [83]:
df["TotalCharges"] = df["TotalCharges"].fillna(0)

In [84]:
print("Missing TotalCharges:", df["TotalCharges"].isnull().sum())

Missing TotalCharges: 0


In [85]:
print("=== DATA VALIDATION ===")

# 1. Missing values
print("\n1. Missing Values:")
print(df.isnull().sum().sum())

# 2. Duplicate rows
print("\n2. Duplicate Rows:")
print(df.duplicated().sum())

# 3. Jumlah baris dan kolom
print("\n3. Dataset Shape:")
print(df.shape)

# 4. Tipe data
print("\n4. Data Types:")
print(df.dtypes)

=== DATA VALIDATION ===

1. Missing Values:
0

2. Duplicate Rows:
0

3. Dataset Shape:
(7043, 20)

4. Data Types:
customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges        float64
dtype: object


---

## Train

In [92]:
X_train, X_test, y_train, y_test = train_test_split(df, y, test_size=0.2, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

In [93]:
num_cols = X_train.select_dtypes("number").columns.tolist()
cat_cols = X_train.select_dtypes("object").columns.tolist()

/tmp/ipykernel_5951/2895117495.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train.select_dtypes("object").columns.tolist()


In [94]:
best = tune(X_train, y_train, num_cols, cat_cols)
print(best)

{'n_estimators': 400, 'learning_rate': 0.013626900549586705, 'num_leaves': 11, 'subsample': 0.6072141486627343, 'colsample_bytree': 0.7975672476485954, 'scale_pos_weight': 3.6672273563573783}


In [ ]:
model = build_pipeline(num_cols, cat_cols, **best)
model.fit(X_train, y_train)
y_pred = model.predict_proba(X_val)[:, 1]
threshold, precision, recall = pick_threshold(y_val, y_pred)
print(f"Threshold: {threshold}, Precision: {precision}, Recall: {recall}")


Threshold: 0.6310452342462821, Precision: 0.569620253164557, Recall: 0.7525083612040134


In [103]:
y_proba = model.predict_proba(X_test)[:, 1]

In [104]:
y_pred = (y_proba > threshold).astype(int)
print(y_pred.shape)
print(y_test.shape)


(1409,)
(1409,)


In [105]:
from sklearn.metrics import classification_report
## Test
y_pred = (y_proba > threshold).astype(int)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.90      0.77      0.83      1035
           1       0.54      0.75      0.63       374

    accuracy                           0.77      1409
   macro avg       0.72      0.76      0.73      1409
weighted avg       0.80      0.77      0.78      1409

